# Day 10 — Week 1–2 Mini Project: Inventory Health Dashboard
**Estimated time:** 75–90 min
**Datasets:** `materials_inventory.csv`, `sales_orders.csv`

## Learning Objectives
- Apply all Week 1–2 skills in an end-to-end analytical workflow
- Build a structured deliverable: dead stock identification, slow-mover analysis, inventory value at risk
- Export findings to CSV and HTML for business distribution

---
*This project mirrors what you would build in a real SAP analytics initiative — an inventory health scorecard combining stock data with demand signals from order history.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
TODAY = pd.Timestamp.today().normalize()

# Load and clean
inv = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
inv["LABST"] = pd.to_numeric(inv["LABST"], errors="coerce").fillna(0)
inv["STPRS"] = pd.to_numeric(inv["STPRS"], errors="coerce").fillna(0)
inv["LAST_MOVEMENT_DATE"] = pd.to_datetime(inv["LAST_MOVEMENT_DATE"], errors="coerce")
inv["MAKTX"] = inv["MAKTX"].astype(str).str.strip().str.upper()
inv["MATNR"] = inv["MATNR"].astype(str).str.zfill(18)
inv["WERKS"] = inv["WERKS"].astype(str).str.zfill(4)

sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
sales["ERDAT"] = pd.to_datetime(sales["ERDAT"], errors="coerce")
sales["NETWR"] = pd.to_numeric(sales["NETWR"], errors="coerce").fillna(0)
sales["MENGE"] = pd.to_numeric(sales["MENGE"], errors="coerce").fillna(0)
sales["MATNR"] = sales["MATNR"].astype(str).str.zfill(18)

print(f"Inventory: {inv.shape}  |  Sales: {sales.shape}")
print(f"Analysis date: {TODAY.date()}")

In [ ]:
# -- TASK 1: Dead stock identification --
# Dead stock = has inventory > 0 but no sales in last 12 months
cutoff_12m = TODAY - pd.DateOffset(months=12)
recently_sold = set(sales.loc[sales["ERDAT"] >= cutoff_12m, "MATNR"].unique())

inv["INV_VALUE"] = inv["LABST"] * inv["STPRS"]
inv["days_no_movement"] = (TODAY - inv["LAST_MOVEMENT_DATE"]).dt.days

dead_stock = inv[(inv["LABST"] > 0) & ~inv["MATNR"].isin(recently_sold)].copy()
dead_stock_summary = (dead_stock
    .groupby(["MATNR","MAKTX","WERKS"])
    .agg(
        total_qty=("LABST","sum"),
        inv_value=("INV_VALUE","sum"),
        days_no_movement=("days_no_movement","max"),
        material_type=("MATERIAL_TYPE","first"),
    )
    .sort_values("inv_value", ascending=False)
    .reset_index()
)

total_dead_value = dead_stock_summary["inv_value"].sum()
print(f"Dead stock materials: {len(dead_stock_summary):,}")
print(f"Total dead stock value: ${total_dead_value:,.2f}")
display(dead_stock_summary.head(15))

In [ ]:
# -- TASK 2: Slow-mover analysis --
inv["aging_bucket"] = pd.cut(
    inv["days_no_movement"].fillna(9999),
    bins=[-1, 29, 89, 179, float("inf")],
    labels=["Active (< 30d)", "Slow (30-90d)", "Very Slow (90-180d)", "Dormant (> 180d)"]
)

slow_mover_profile = (
    inv[inv["LABST"] > 0]
    .groupby("aging_bucket", observed=True)
    .agg(
        n_materials=("MATNR","nunique"),
        total_qty=("LABST","sum"),
        total_value=("INV_VALUE","sum"),
    )
    .assign(value_pct=lambda d: d["total_value"] / d["total_value"].sum() * 100)
    .reset_index()
)
print("Slow-mover analysis:")
display(slow_mover_profile)

In [ ]:
# -- TASK 3: Inventory value at risk --
# At risk = slow/dormant stock (> 90 days no movement) + no recent sales
at_risk = inv[
    (inv["LABST"] > 0) &
    (inv["days_no_movement"].fillna(9999) > 90) &
    (~inv["MATNR"].isin(recently_sold))
].copy()

at_risk_value = at_risk["INV_VALUE"].sum()
total_inv_value = inv["INV_VALUE"].sum()
risk_pct = at_risk_value / total_inv_value * 100 if total_inv_value > 0 else 0

print(f"Inventory value at risk: ${at_risk_value:,.2f}")
print(f"As % of total inventory: {risk_pct:.1f}%")

at_risk_by_type = (at_risk
    .groupby("MATERIAL_TYPE")
    .agg(n_mats=("MATNR","nunique"), value=("INV_VALUE","sum"))
    .sort_values("value", ascending=False))
display(at_risk_by_type)

In [ ]:
# -- TASK 4: Top / Bottom performers by sales velocity --
velocity = (sales[sales["ERDAT"] >= cutoff_12m]
            .groupby("MATNR")
            .agg(total_sold_qty=("MENGE","sum"), total_revenue=("NETWR","sum"),
                 order_count=("VBELN","count"))
            .reset_index())

perf = pd.merge(inv[["MATNR","MAKTX","LABST","INV_VALUE","MATERIAL_TYPE"]].drop_duplicates("MATNR"),
                velocity, on="MATNR", how="left")
perf["total_sold_qty"] = perf["total_sold_qty"].fillna(0)
perf["total_revenue"]  = perf["total_revenue"].fillna(0)

top10_performers    = perf.sort_values("total_revenue", ascending=False).head(10)
bottom10_performers = perf[perf["LABST"] > 0].sort_values("total_sold_qty").head(10)

print("Top 10 by revenue (last 12m):")
display(top10_performers[["MATNR","MAKTX","total_sold_qty","total_revenue","order_count"]])
print("\nBottom 10 (has stock but lowest sales velocity):")
display(bottom10_performers[["MATNR","MAKTX","LABST","INV_VALUE","total_sold_qty"]])

In [ ]:
%%time
# -- Executive summary DataFrame --
summary = pd.DataFrame({
    "metric": [
        "Total materials in inventory",
        "Total inventory value",
        "Dead stock materials (no sales 12m)",
        "Dead stock value",
        "Dead stock % of total value",
        "At-risk materials (>90d no mvmt, no recent sales)",
        "At-risk value",
        "At-risk % of total value",
    ],
    "value": [
        f"{inv['MATNR'].nunique():,}",
        f"${total_inv_value:,.2f}",
        f"{len(dead_stock_summary):,}",
        f"${total_dead_value:,.2f}",
        f"{total_dead_value/total_inv_value*100:.1f}%" if total_inv_value else "N/A",
        f"{at_risk['MATNR'].nunique():,}",
        f"${at_risk_value:,.2f}",
        f"{risk_pct:.1f}%",
    ]
})
display(summary)

In [ ]:
# -- Charts --
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Inventory Health Dashboard", fontsize=15, fontweight="bold")

# Chart 1: Inventory value by aging bucket
ax = axes[0]
colors = ["#16A34A","#F59E0B","#F97316","#DC2626"]
if not slow_mover_profile.empty:
    bars = ax.bar(slow_mover_profile["aging_bucket"],
                  slow_mover_profile["total_value"],
                  color=colors[:len(slow_mover_profile)], edgecolor="white")
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h * 1.01, f"${h:,.0f}",
                ha="center", va="bottom", fontsize=8)
ax.set_title("Inventory Value by Aging Bucket")
ax.set_ylabel("Total Value ($)")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", linestyle="--", alpha=0.4)

# Chart 2: Top 10 performers by revenue
ax = axes[1]
top10_performers.sort_values("total_revenue").tail(10).plot.barh(
    x="MATNR", y="total_revenue", ax=ax, color="#2563EB", legend=False)
ax.set_title("Top 10 Materials by Revenue (Last 12m)")
ax.set_xlabel("Revenue ($)")
ax.set_ylabel("")
ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("day10_inventory_health.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved.")

In [ ]:
# -- Export results --
dead_stock_summary.to_csv("day10_dead_stock.csv", index=False)
print("Saved: day10_dead_stock.csv")

html_table = dead_stock_summary.head(30).to_html(index=False, float_format="{:,.2f}".format)
html_out = (
    "<!DOCTYPE html><html><head>"
    "<style>"
    "body { font-family: Arial, sans-serif; margin: 20px; }"
    "h2 { color: #1e3a5f; }"
    "table { border-collapse: collapse; width: 100%; }"
    "th { background: #1e3a5f; color: white; padding: 8px; text-align: left; }"
    "td { padding: 6px 8px; border-bottom: 1px solid #ddd; }"
    "tr:nth-child(even) { background: #f5f5f5; }"
    "</style></head><body>"
    f"<h2>Dead Stock Report</h2>"
    f"<p>Materials with inventory on hand but no sales in the last 12 months.</p>"
    f"{html_table}"
    "</body></html>"
)
with open("day10_dead_stock_report.html", "w") as f:
    f.write(html_out)
print("Saved: day10_dead_stock_report.html")

---
## Self-Grade Rubric

| Dimension | 1 — Needs Work | 2 — Functional | 3 — Solid | 4 — Strong |
|-----------|---------------|----------------|-----------|------------|
| **Dead stock logic** | Hard-coded dates | Correct filter | Parameterized | + business impact explained |
| **Slow-mover buckets** | Only 1–2 buckets | All 4 correct | + aging value % | + trend insight |
| **Value at risk** | Rough estimate | Correct calc | + by material type | + writeback-ready format |
| **Code quality** | Mostly working | PEP 8 compliant | Reusable functions | + docstrings + type hints |
| **Charts** | Basic plot | Labeled axes | + formatted values | + annotations + saved PNG |
| **Exports** | None | CSV only | CSV + HTML | + presentation-quality |

**Target:** Score 3 across all dimensions before moving to Week 3.

---
## Stretch Challenges

1. Wrap everything in `inventory_health_report(plant=None, as_of_date=None)` — fully parameterized
2. Write-off recommendation list: dead stock > 180 days + value < $100 → automatic write-off candidates
3. ABC analysis overlay: classify materials into A (top 80% of value), B (next 15%), C (bottom 5%) and cross-tab with aging buckets